In [1]:
!pip install colorama

In [ ]:
import os
import pandas as pd
import numpy as np
import math
import datetime as dt

from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance, accuracy_score
from sklearn.preprocessing import MinMaxScaler

from itertools import cycle
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import seaborn as sns
import matplotlib.pyplot as plt
from colorama import Fore


In [ ]:
"""
=============================================================================
ENTREGA 2 — Análisis de Datos Financieros y Segmentación de Clientes
European Market Research & Finance Lab (EMRFL)
=============================================================================
Instrucciones:
  1. Instalar dependencias:  pip install yfinance pandas numpy matplotlib seaborn scikit-learn
  2. Ejecutar el script completo:  python analisis_financiero_entrega2.py
  3. Los gráficos se guardan como PNG y están listos para incluir en la memoria.

Nota: En un entorno con acceso a internet, los datos se descargan automáticamente
      con yfinance. Si no hay conexión, se generan datos simulados realistas.
=============================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────
# SECCIÓN 1 — DESCARGA Y PREPARACIÓN DE DATOS FINANCIEROS
# ─────────────────────────────────────────────────────────────────

def descargar_datos(tickers, start='2023-01-01', end='2024-12-31'):
    """
    Intenta descargar datos reales con yfinance.
    Si falla, genera datos simulados con parámetros realistas.
    """
    try:
        import yfinance as yf
        data = {}
        for ticker in tickers:
            df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
            data[ticker] = df['Close']
        prices = pd.DataFrame(data)
        print("✓ Datos reales descargados con yfinance")
        return prices
    except Exception as e:
        print(f"⚠ yfinance no disponible ({e}). Usando datos simulados.")
        return _simular_datos(start, end)

def _simular_datos(start, end):
    """Genera datos simulados con parámetros estadísticos realistas (2023-2024)."""
    np.random.seed(42)
    dates = pd.date_range(start, end, freq='B')
    n = len(dates)

    def _asset(start_price, annual_return, annual_vol):
        daily_ret = annual_return / 252
        daily_vol = annual_vol / np.sqrt(252)
        r = np.random.normal(daily_ret, daily_vol, n)
        return start_price * np.exp(np.cumsum(r))

    # Parámetros basados en datos históricos 2023-2024
    prices = pd.DataFrame({
        'BTC-USD': _asset(16_500, 0.70, 0.70),   # Bitcoin
        'ETH-USD': _asset(1_200,  0.55, 0.75),    # Ethereum
        'SPY':     _asset(380,    0.24, 0.16),     # S&P 500 ETF
    }, index=dates)
    return prices

# Activos a analizar
TICKERS = ['BTC-USD', 'ETH-USD', 'SPY']
LABELS  = {'BTC-USD': 'Bitcoin (BTC)', 'ETH-USD': 'Ethereum (ETH)', 'SPY': 'S&P 500 ETF (SPY)'}
COLORS  = {'BTC-USD': '#F7931A', 'ETH-USD': '#627EEA', 'SPY': '#00C49A'}

prices = descargar_datos(TICKERS)
print(f"\nPeriodo: {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Sesiones: {len(prices)}\n")

# Limpieza: eliminar filas con NaN
prices = prices.dropna()

# Rendimientos diarios
returns = prices.pct_change().dropna()

# ─────────────────────────────────────────────────────────────────
# SECCIÓN 2 — ESTADÍSTICAS DESCRIPTIVAS
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("RESUMEN ESTADÍSTICO DE ACTIVOS")
print("=" * 60)

for col in TICKERS:
    r = returns[col]
    p = prices[col]
    vol_anual = r.std() * np.sqrt(252)
    retorno   = (p.iloc[-1] / p.iloc[0]) - 1
    sharpe    = (r.mean() * 252) / vol_anual
    max_dd    = (p / p.cummax() - 1).min()

    print(f"\n▸ {LABELS[col]}")
    print(f"  Precio inicial : ${p.iloc[0]:>10,.2f}")
    print(f"  Precio final   : ${p.iloc[-1]:>10,.2f}")
    print(f"  Retorno total  : {retorno*100:>+8.2f}%")
    print(f"  Vol. anualizada: {vol_anual*100:>8.2f}%")
    print(f"  Sharpe ratio   : {sharpe:>8.2f}")
    print(f"  Máx. drawdown  : {max_dd*100:>+8.2f}%")

# ─────────────────────────────────────────────────────────────────
# SECCIÓN 3 — VISUALIZACIONES
# ─────────────────────────────────────────────────────────────────

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

# --- Figura 1: Evolución del precio (3 paneles) ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), facecolor='#F8F9FA')
fig.suptitle('Evolución del Precio — Bitcoin, Ethereum y S&P 500 ETF\n(Enero 2023 – Diciembre 2024)',
             fontsize=15, fontweight='bold')

for ax, col in zip(axes, TICKERS):
    ax.fill_between(prices.index, prices[col], alpha=0.15, color=COLORS[col])
    ax.plot(prices.index, prices[col], color=COLORS[col], lw=1.8, label=LABELS[col])
    ax.set_ylabel('Precio (USD)', fontsize=10)
    ax.legend(loc='upper left', fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[-1].set_xlabel('Fecha', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig1_precios.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✓ fig1_precios.png guardado")

# --- Figura 2: Comparativa normalizada ---
prices_norm = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 6), facecolor='#F8F9FA')
for col in TICKERS:
    ax.plot(prices_norm.index, prices_norm[col], color=COLORS[col], lw=2.2, label=LABELS[col])
ax.axhline(100, color='gray', lw=1, linestyle='--', alpha=0.6)
ax.set_ylabel('Rentabilidad acumulada (base 100)', fontsize=11)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_title('Comparativa Normalizada de Activos (Base = 100 en enero 2023)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('fig2_comparativa.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ fig2_comparativa.png guardado")

# --- Figura 3: Rendimientos diarios ---
fig, axes = plt.subplots(3, 1, figsize=(14, 9), facecolor='#F8F9FA')
fig.suptitle('Rendimientos Diarios por Activo', fontsize=14, fontweight='bold')

for ax, col in zip(axes, TICKERS):
    pos = returns[col].clip(lower=0)
    neg = returns[col].clip(upper=0)
    ax.bar(returns.index, pos, color=COLORS[col], alpha=0.7, width=1)
    ax.bar(returns.index, neg, color='#E63946', alpha=0.7, width=1)
    ax.set_title(LABELS[col], fontsize=11, color=COLORS[col], fontweight='bold')
    ax.set_ylabel('Rendimiento', fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))

axes[-1].set_xlabel('Fecha', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig3_rendimientos.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ fig3_rendimientos.png guardado")

# --- Figura 4: Volatilidad (media móvil 30 días) ---
vol = returns.rolling(30).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 6), facecolor='#F8F9FA')
for col in TICKERS:
    ax.plot(vol.index, vol[col], color=COLORS[col], lw=2, label=LABELS[col])
ax.set_ylabel('Volatilidad anualizada (media móvil 30 días)', fontsize=11)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_title('Análisis de Volatilidad por Activo', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('fig4_volatilidad.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ fig4_volatilidad.png guardado")

# ─────────────────────────────────────────────────────────────────
# SECCIÓN 4 — PREDICCIONES Y MAE
# ─────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("PREDICCIONES CON MODELO DE PASEO ALEATORIO (BITCOIN)")
print("=" * 60)

p_btc = prices['BTC-USD'].copy()
horizons = [1, 5, 10, 15]
maes = {}

for h in horizons:
    # Modelo naive: predicción = precio actual → precio en t+h
    actual    = p_btc.shift(-h).dropna()
    predicted = p_btc.loc[actual.index]
    mae_abs   = (actual - predicted).abs().mean()
    mae_pct   = mae_abs / predicted.mean() * 100
    maes[h]   = mae_pct
    print(f"  Horizonte {h:2d} días → MAE = {mae_pct:.2f}%")

print("\n▸ Interpretación: El error crece porque la varianza del proceso")
print("  de paseo aleatorio se acumula proporcionalmente al horizonte.")
print("  A 15 días, el error supera el 13%, haciendo la predicción")
print("  poco útil para decisiones de inversión concretas.")

# --- Figura 5: MAE + ejemplo visual ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#F8F9FA')

bars = axes[0].bar(horizons, list(maes.values()),
                   color=['#F7931A', '#627EEA', '#00C49A', '#E63946'],
                   width=2.5, edgecolor='white')
axes[0].set_xlabel('Horizonte de predicción (días)', fontsize=11)
axes[0].set_ylabel('MAE (%)', fontsize=11)
axes[0].set_title('Error Medio Absoluto (MAE)\nModelo Paseo Aleatorio — Bitcoin', fontsize=12, fontweight='bold')
axes[0].set_xticks(horizons)
for bar, v in zip(bars, maes.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{v:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Ejemplo visual predicción a 15 días
p_sub = p_btc.iloc[-75:]
actual_future = p_btc.iloc[-15:]
naive_val = [p_btc.iloc[-16]] * 15

axes[1].plot(p_sub.index[:-15], p_sub.iloc[:-15], color='#F7931A', lw=2, label='Histórico')
axes[1].plot(actual_future.index, actual_future.values, color='#1A1A2E', lw=2, linestyle='--', label='Real')
axes[1].plot(actual_future.index, naive_val, color='#E63946', lw=2, linestyle=':', label='Predicción naive')
axes[1].fill_between(actual_future.index, actual_future.values, naive_val, alpha=0.15, color='#E63946')
axes[1].set_title('Predicción Naive a 15 días — Bitcoin', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fecha', fontsize=11)
axes[1].set_ylabel('Precio (USD)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('fig5_predicciones.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ fig5_predicciones.png guardado")

# ─────────────────────────────────────────────────────────────────
# SECCIÓN 5 — SEGMENTACIÓN DE CLIENTES
# ─────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("SEGMENTACIÓN DE CLIENTES")
print("=" * 60)

np.random.seed(99)
N = 500

# Segmento 1: Especulador Digital (joven, alta tolerancia al riesgo, muy digital)
n1 = 160
# Segmento 2: Inversor Pragmático (mediana edad, riesgo moderado)
n2 = 200
# Segmento 3: Conservador Experimentado (mayor, bajo riesgo, poco digital)
n3 = 140

ages   = np.concatenate([np.random.normal(28, 4, n1), np.random.normal(42, 6, n2), np.random.normal(57, 5, n3)])
income = np.concatenate([np.random.normal(28000, 5000, n1), np.random.normal(52000, 8000, n2), np.random.normal(75000, 12000, n3)])
exp    = np.concatenate([np.random.choice([1,2], n1), np.random.choice([2,3,4], n2), np.random.choice([3,4,5], n3)])
risk   = np.concatenate([np.random.choice([4,5], n1), np.random.choice([2,3], n2), np.random.choice([1,2], n3)])
digital= np.concatenate([np.random.normal(8.5,1,n1), np.random.normal(5.5,1.5,n2), np.random.normal(3.2,1.5,n3)])
segs   = ['Especulador Digital']*n1 + ['Inversor Pragmático']*n2 + ['Conservador Experimentado']*n3

customers = pd.DataFrame({
    'Edad':                  ages.clip(18, 75).astype(int),
    'Ingresos_Anuales':      income.clip(15000, 120000).round(-2),
    'Experiencia_Financiera':exp.clip(1, 5).astype(int),
    'Tolerancia_Riesgo':     risk.clip(1, 5).astype(int),
    'Uso_Plataformas_Digital':digital.clip(1,10).round(1),
    'Segmento':              segs,
})

customers.to_csv('customers.csv', index=False)
print("✓ Dataset guardado: customers.csv")
print(f"  Total clientes: {len(customers)}")
print("\nDistribución por segmento:")
print(customers['Segmento'].value_counts().to_string())

print("\nEstadísticas descriptivas por segmento:")
print(customers.groupby('Segmento')[['Edad','Ingresos_Anuales','Experiencia_Financiera',
                                      'Tolerancia_Riesgo','Uso_Plataformas_Digital']].mean().round(1))

# --- Figura 6: Análisis descriptivo segmentación ---
SEG_C = {
    'Especulador Digital':       '#F7931A',
    'Inversor Pragmático':       '#627EEA',
    'Conservador Experimentado': '#00C49A',
}
order = list(SEG_C.keys())

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='#F8F9FA')
fig.suptitle('Análisis Descriptivo y Segmentación de Clientes', fontsize=15, fontweight='bold')

# Histograma de edades
for seg, grp in customers.groupby('Segmento'):
    axes[0,0].hist(grp['Edad'], bins=20, alpha=0.6, label=seg, color=SEG_C[seg], edgecolor='white')
axes[0,0].set_xlabel('Edad')
axes[0,0].set_ylabel('Frecuencia')
axes[0,0].set_title('Distribución de Edad por Segmento', fontweight='bold')
axes[0,0].legend(fontsize=9)

# Boxplot de ingresos
data_bp = [customers[customers['Segmento']==s]['Ingresos_Anuales'].values for s in order]
bp = axes[0,1].boxplot(data_bp, patch_artist=True, labels=['Especulador\nDigital','Inversor\nPragmático','Conservador\nExperimentado'])
for patch, seg in zip(bp['boxes'], order):
    patch.set_facecolor(SEG_C[seg]); patch.set_alpha(0.7)
axes[0,1].set_ylabel('Ingresos Anuales (€)')
axes[0,1].set_title('Distribución de Ingresos por Segmento', fontweight='bold')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Scatter: riesgo vs digital
for seg, grp in customers.groupby('Segmento'):
    axes[1,0].scatter(grp['Tolerancia_Riesgo'], grp['Uso_Plataformas_Digital'],
                      alpha=0.35, color=SEG_C[seg], label=seg, s=40, edgecolors='none')
axes[1,0].set_xlabel('Tolerancia al Riesgo (1=Bajo, 5=Alto)')
axes[1,0].set_ylabel('Uso de Plataformas Digitales (1-10)')
axes[1,0].set_title('Tolerancia al Riesgo vs. Uso Digital', fontweight='bold')
axes[1,0].legend(fontsize=9)

# Perfil comparativo normalizado
vars_ = ['Edad','Experiencia_Financiera','Tolerancia_Riesgo','Uso_Plataformas_Digital']
means = customers.groupby('Segmento')[vars_].mean()
means_norm = (means - means.min()) / (means.max() - means.min())
x = np.arange(len(vars_))
w = 0.25
for i, seg in enumerate(order):
    axes[1,1].bar(x + i*w, means_norm.loc[seg], w, label=seg, color=SEG_C[seg], alpha=0.8)
axes[1,1].set_xticks(x + w)
axes[1,1].set_xticklabels(['Edad','Experiencia','Riesgo','Digital'], fontsize=9)
axes[1,1].set_ylabel('Valor normalizado (0–1)')
axes[1,1].set_title('Perfil Comparativo de Segmentos\n(variables normalizadas)', fontweight='bold')
axes[1,1].legend(fontsize=9)

plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig('fig6_segmentacion.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ fig6_segmentacion.png guardado")

print("\n" + "=" * 60)
print("ANÁLISIS COMPLETADO — ENTREGA 2")
print("Archivos generados:")
print("  • fig1_precios.png         → Evolución del precio")
print("  • fig2_comparativa.png     → Comparativa normalizada")
print("  • fig3_rendimientos.png    → Rendimientos diarios")
print("  • fig4_volatilidad.png     → Volatilidad (MM30)")
print("  • fig5_predicciones.png    → MAE y predicción naive")
print("  • fig6_segmentacion.png    → Segmentación de clientes")
print("  • customers.csv            → Dataset de clientes")
print("=" * 60)

In [ ]:
from IPython.display import Image, display

figuras = [
    ('fig1_precios.png',      'Evolución del precio'),
    ('fig2_comparativa.png',  'Comparativa normalizada (base 100)'),
    ('fig3_rendimientos.png', 'Rendimientos diarios'),
    ('fig4_volatilidad.png',  'Volatilidad anualizada (MM30)'),
    ('fig5_predicciones.png', 'MAE y predicción naive'),
    ('fig6_segmentacion.png', 'Segmentación de clientes'),
]

for archivo, titulo in figuras:
    print(f'\n── {titulo}')
    display(Image(archivo, width=800))

In [ ]:
!pip install yfinance -q

import yfinance as yf
import pandas as pd

tickers = ['BTC-USD', 'ETH-USD', 'SPY']
frames = {}
for t in tickers:
    df = yf.download(t, start='2023-01-01', end='2024-12-31', progress=False, auto_adjust=True)
    frames[t] = df['Close'].squeeze()  # <-- esto arregla el error del índice

prices = pd.DataFrame(frames)
prices = prices.dropna()
print("✓ Datos reales descargados")
print(prices.tail(3))